<a href="https://colab.research.google.com/github/ysuter/FHNW-BAI-DeepLearning/blob/main/W09_RNN_Zeitreihen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/ysuter/FHNW-BAI-DeepLearning/blob/main/W09_RNN_Zeitreihen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="In Colab öffnen"/></a>

# W9 - Deep Learning für Zeitreihen: Rekurrente Neuronale Netze

Wir sagen die stündliche Nachfrage eines Velo-Verleihsystems voraus. Besonderer Fokus auf das Splittijng der Daten.

**Lerninhalte**

1. Zeitreihendaten für ein neuronales Netz in Fenster aufteilen
2. Einen korrekten zeitlichen Split bauen - und erklären, warum ein zufälliger Split keine realitische Resultate liefert
3. Baselines rechnen und ein Modell daran messen
4. Ein GRU/LSTM in PyTorch trainieren und das Resultat einordnen

**Laufzeit:** Läuft auch auf der CPU :)

## 0. Setup

Wir fixieren alle Zufallsgeneratoren. Ein fixer Seed gibt auf der gleichen Maschine den selben Output.

In [ ]:
import numpy as np, pandas as pd, torch, torch.nn as nn, time
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

plt.rcParams["figure.figsize"] = (11, 3.5)
print("PyTorch", torch.__version__)

## 1. Daten laden

Der Datensatz *Bike Sharing* (UCI) enthält die stündliche Anzahl Ausleihen eines
Velo-Verleihsystems in Washington D.C. für 2011 und 2012, zusammen mit Wetter- und
Kalenderangaben.

In [ ]:
URLS = [
    "https://raw.githubusercontent.com/KeithJLZ/UCI-Bike-Sharing-Dataset/main/hour.csv",
    "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip",
]

def lade_daten():
    import io, zipfile, urllib.request
    for url in URLS:
        try:
            roh = urllib.request.urlopen(url, timeout=30).read()
            if url.endswith(".zip"):
                with zipfile.ZipFile(io.BytesIO(roh)) as z:
                    return pd.read_csv(z.open("hour.csv"))
            return pd.read_csv(io.BytesIO(roh))
        except Exception as e:
            print(f"  ... {url.split('/')[2]} nicht erreichbar ({type(e).__name__})")
    raise RuntimeError("Keine Quelle erreichbar - lade hour.csv manuell hoch.")

df = lade_daten()
df["ts"] = pd.to_datetime(df["dteday"]) + pd.to_timedelta(df["hr"], unit="h")
df = df.set_index("ts").sort_index()
print(f"{len(df)} Zeilen von {df.index.min()} bis {df.index.max()}")
df[["season", "workingday", "weathersit", "temp", "cnt"]].head()

### Lücken schliessen

Eine Zeitreihe braucht ein regelmässiges Raster. Fehlt eine Stunde, stimmt `y[t-24]`
nicht mehr mit "gestern zur selben Zeit" überein. Wir legen die Daten deshalb auf ein
vollständiges Stundenraster und füllen die Lücken.

Achtung: interpolierte Zielwerte sind erfundene Daten. Bei 165 von 17'544 Stunden ist das
vertretbar - muss aber bei einer realen Anwendung sorgfältig untersucht werden.

In [ ]:
voll = pd.date_range(df.index.min(), df.index.max(), freq="h")
print(f"Fehlende Stunden: {len(voll) - len(df)} von {len(voll)}")

df = df.reindex(voll)
for c in ["cnt", "temp", "atemp", "hum", "windspeed"]:
    df[c] = df[c].interpolate(limit_direction="both")
for c in ["holiday", "workingday", "weathersit"]:
    df[c] = df[c].ffill()
df.index.name = "ts"
print("Fehlende Werte nach dem Füllen:", int(df["cnt"].isna().sum()))

### Erst schauen, dann modellieren

Zwei Plots um einen ersten Eindruck der Daten zu erhalten.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 6))

df["cnt"].loc["2012-06-04":"2012-06-24"].plot(ax=ax[0], lw=1)
ax[0].set_title("Drei Wochen im Juni 2012 - Tagesrhythmus und Wochenenden")
ax[0].set_ylabel("Ausleihen / Stunde")

profil = df.groupby([df["workingday"], df.index.hour])["cnt"].mean().unstack(0)
profil.columns = ["Wochenende / Feiertag", "Arbeitstag"]
profil.plot(ax=ax[1])
ax[1].set_title("Mittlerer Tagesverlauf")
ax[1].set_xlabel("Stunde"); ax[1].set_ylabel("Ausleihen")
plt.tight_layout(); plt.show()

Der untere Plot zeigt das Wichtigste: An Arbeitstagen gibt es zwei Pendlerspitzen
(ca. 8 und 17 Uhr), am Wochenende einen breiten Mittagsberg. Ein Modell, das nur die
Tageszeit kennt, kommt damit schon weit.

**Merke das für später:** Was wir hier von blossem Auge sehen, muss ein Modell erst
einmal erreichen.

## 2. Features bauen - und dabei nicht schummeln

Die entscheidende Frage bei jedem Feature lautet: **Kenne ich diesen Wert zum Zeitpunkt
der Vorhersage schon?**

Wir sagen den Wert zur Stunde `t` voraus, und wir stehen dabei am Ende von Stunde `t-1`.

| Feature | Verfügbar? | Begründung |
|---|---|---|
| Stunde, Wochentag, Monat, Feiertag | ja | Der Kalender ist im Voraus bekannt |
| `y[t-1]`, `y[t-24]`, `y[t-168]` | ja | Vergangene Ausleihen |
| Temperatur **zur Stunde t** | **nein** | Der Istwert existiert erst hinterher |
| Temperatur zur Stunde t-1 | ja | Bereits gemessen |

Deshalb gehen alle Wettergrössen nur **verzögert** ins Modell. Wer den Istwert nimmt,
baut sich ein Modell, das im Betrieb nicht eingesetzt werden kann.

In [ ]:
d = pd.DataFrame(index=df.index)
d["y"] = df["cnt"].astype(float)

# Kalender - im Voraus bekannt
d["hour"] = d.index.hour
d["dow"] = d.index.dayofweek
d["month"] = d.index.month
d["workingday"] = df["workingday"].astype(float)
d["holiday"] = df["holiday"].astype(float)

# Lags - nur Vergangenheit
for L in [1, 2, 3, 24, 25, 168]:
    d[f"lag{L}"] = d["y"].shift(L)
d["roll24"] = d["y"].shift(1).rolling(24).mean()
d["roll168"] = d["y"].shift(1).rolling(168).mean()

# Wetter - nur verzögert
for c in ["temp", "atemp", "hum", "windspeed"]:
    d[f"{c}_lag1"] = df[c].astype(float).shift(1)

d = d.dropna()
FEATURES = [c for c in d.columns if c != "y"]
print(f"{len(d)} Zeilen, {len(FEATURES)} Features")
print(FEATURES)

### Der zeitliche Split

Train ist Vergangenheit, Validation die Mitte, Test die Zukunft. Kein Punkt aus der
Zukunft darf ins Training.

In [ ]:
TRAIN_ENDE = "2012-06-30 23:00"
VAL_ENDE   = "2012-09-30 23:00"

train = d.loc[:TRAIN_ENDE]
val   = d.loc["2012-07-01":VAL_ENDE]
test  = d.loc["2012-10-01":]

for name, teil in [("Train", train), ("Val", val), ("Test", test)]:
    print(f"{name:6s} {len(teil):6d} Zeilen   {teil.index.min().date()} bis {teil.index.max().date()}")

plt.figure(figsize=(11, 2.6))
for teil, farbe, name in [(train, "#4C78A8", "Train"), (val, "#F58518", "Val"), (test, "#54A24B", "Test")]:
    plt.plot(teil.index, teil["y"], lw=0.4, color=farbe, label=name)
plt.legend(loc="upper left"); plt.title("Zeitlicher Split"); plt.ylabel("Ausleihen")
plt.tight_layout(); plt.show()

## 3. Die Falle: zufälliger gegen zeitlichen Split

Jetzt der wichtigste Moment dieses Notebooks. Wir nehmen **dasselbe Modell** und
**dieselben Daten** - und ändern nur, wie wir splitten.

In [ ]:
X, y = d[FEATURES].values, d["y"].values
grenze = int(0.8 * len(d))

# Variante A: zufällig mischen, dann splitten
rng = np.random.default_rng(SEED)
perm = rng.permutation(len(d))
mA = HistGradientBoostingRegressor(random_state=SEED, max_iter=200)
mA.fit(X[perm[:grenze]], y[perm[:grenze]])
rmse_zufall = rmse(y[perm[grenze:]], mA.predict(X[perm[grenze:]]))

# Variante B: zeitlich splitten
mB = HistGradientBoostingRegressor(random_state=SEED, max_iter=200)
mB.fit(X[:grenze], y[:grenze])
rmse_zeitlich = rmse(y[grenze:], mB.predict(X[grenze:]))

print(f"Zufälliger Split : RMSE = {rmse_zufall:6.1f}")
print(f"Zeitlicher  Split : RMSE = {rmse_zeitlich:6.1f}")
print(f"\nDer zufällige Split sieht um {100*(1-rmse_zufall/rmse_zeitlich):.0f}% besser aus.")

### Welche der beiden Zahlen würdest du einem Kunden zeigen?

Der zufällige Split ist aus zwei unabhängigenGründen zu optimistisch:

- ...
- ...

## 4. Baselines - bevor irgendein Netz trainiert wird

Zwei Baselines kosten zusammen zwei Zeilen Code und entscheiden darüber, ob dein
Resultat später etwas bedeutet.

* **Naiv:** `y[t] = y[t-1]` - "gleich viel wie in der letzten Stunde"
* **Saisonal-naiv:** `y[t] = y[t-24]` - "gleich viel wie gestern zu dieser Zeit"

In [ ]:
ergebnisse = {}

def bewerte(name, y_true, y_pred):
    ergebnisse[name] = {"MAE": mean_absolute_error(y_true, y_pred), "RMSE": rmse(y_true, y_pred)}
    print(f"{name:32s} MAE {ergebnisse[name]['MAE']:7.1f}   RMSE {ergebnisse[name]['RMSE']:7.1f}")

bewerte("Naiv  y[t-1]",          test["y"], test["lag1"])
bewerte("Saisonal-naiv  y[t-24]", test["y"], test["lag24"])

### Ein überraschendes Zwischenergebnis

Die **naive** Baseline ist hier *besser* als die saisonal-naive. Das ist kein Fehler,
sondern eine Eigenschaft der Aufgabe: Wir sagen nur **eine Stunde** voraus, und da ist
der letzte beobachtete Wert extrem informativ.

Dreh den Horizont auf 24 Stunden, und das Verhältnis kippt - dann weisst du `y[t-1]`
gar nicht mehr, und die saisonale Baseline gewinnt.

**Daraus folgt:** Die richtige Baseline hängt vom Vorhersagehorizont ab. Rechne beide
und nimm die stärkere als Messlatte. Hier sind das rund **119 RMSE**.

## 5. Ein starker klassischer Vergleichspunkt

Bevor das RNN kommt: Gradient Boosting auf denselben Lag-Features. Das ist in der Praxis
der Gegner, den ein Deep-Learning-Modell schlagen muss - nicht die naive Baseline.

In [ ]:
t0 = time.time()
gbm = HistGradientBoostingRegressor(random_state=SEED, max_iter=400,
                                    learning_rate=0.06, early_stopping=False)
gbm.fit(train[FEATURES], train["y"])
zeit_gbm = time.time() - t0

bewerte("Gradient Boosting", test["y"], gbm.predict(test[FEATURES]))
print(f"\nTrainingszeit: {zeit_gbm:.1f} Sekunden")

## 6. Jetzt das rekurrente Netz

### Von der Tabelle zu Fenstern

Ein RNN bekommt keine Feature-Zeile, sondern eine **Sequenz**. Aus der Zeitreihe
schneiden wir überlappende Fenster der Länge `SEQ`:

```
Fenster 1: [t-48 ... t-1]  ->  y[t]
Fenster 2: [t-47 ... t  ]  ->  y[t+1]
```

Die Form, die PyTorch erwartet, ist **(Batch, Zeit, Features)**. Das ist die häufigste
Fehlerquelle beim Einstieg - wenn eine Fehlermeldung über Dimensionen kommt, ist es fast
immer diese Reihenfolge.

### Zyklische Kodierung

Die Stunde geben wir nicht als Zahl 0-23 weiter, sondern als Sinus/Kosinus-Paar. Sonst
wären 23 Uhr und 0 Uhr maximal weit voneinander entfernt, obwohl sie benachbart sind.

In [ ]:
SEQ = 48   # Fensterlänge in Stunden

s = pd.DataFrame(index=d.index)
s["y"] = d["y"]
s["hour_sin"] = np.sin(2*np.pi*d["hour"]/24)
s["hour_cos"] = np.cos(2*np.pi*d["hour"]/24)
s["dow_sin"]  = np.sin(2*np.pi*d["dow"]/7)
s["dow_cos"]  = np.cos(2*np.pi*d["dow"]/7)
s["workingday"] = d["workingday"]
s["temp_lag1"]  = d["temp_lag1"]
SEQ_COLS = list(s.columns)

# Skalierung: fit NUR auf Train, dann auf alles anwenden
scaler_x = StandardScaler().fit(s.loc[:TRAIN_ENDE][SEQ_COLS].values)
scaler_y = StandardScaler().fit(d.loc[:TRAIN_ENDE][["y"]].values)

Z = scaler_x.transform(s[SEQ_COLS].values).astype(np.float32)
Yz = scaler_y.transform(d[["y"]].values).astype(np.float32).ravel()
idx = d.index

def fenster(von, bis):
    pos = np.where((idx >= von) & (idx <= bis))[0]
    pos = pos[pos >= SEQ]
    return np.stack([Z[i-SEQ:i] for i in pos]), Yz[pos], pos

Xtr, ytr, _    = fenster(idx[0], pd.Timestamp(TRAIN_ENDE))
Xva, yva, _    = fenster(pd.Timestamp("2012-07-01"), pd.Timestamp(VAL_ENDE))
Xte, yte, pos_te = fenster(pd.Timestamp("2012-10-01"), idx[-1])

print(f"Train {Xtr.shape}   Val {Xva.shape}   Test {Xte.shape}")
print("    ^ (Batch, Zeit, Features)")

### Das Modell

`nn.GRU` und `nn.LSTM` haben in PyTorch dieselbe Schnittstelle - wir können also
dieselbe Klasse für beide benutzen und nur die Zelle austauschen.

Wir nehmen den verborgenen Zustand des **letzten** Zeitschritts (`out[:, -1, :]`) und
schicken ihn durch eine lineare Schicht. Das ist der many-to-one-Fall von den Folien.

In [ ]:
class Prognosenetz(nn.Module):
    def __init__(self, n_features, hidden=64, zelle="gru"):
        super().__init__()
        Zelle = nn.GRU if zelle == "gru" else nn.LSTM
        self.rnn  = Zelle(n_features, hidden, num_layers=1, batch_first=True)
        self.kopf = nn.Linear(hidden, 1)

    def forward(self, x):
        out, _ = self.rnn(x)          # out: (Batch, Zeit, Hidden)
        return self.kopf(out[:, -1, :]).squeeze(-1)   # nur letzter Zeitschritt


def trainiere(zelle, epochen=12, lr=1e-3, batch=128):
    torch.manual_seed(SEED)
    netz = Prognosenetz(Xtr.shape[2], hidden=64, zelle=zelle)
    opt  = torch.optim.Adam(netz.parameters(), lr=lr)
    verlust = nn.MSELoss()

    lader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(torch.tensor(Xtr), torch.tensor(ytr)),
        batch_size=batch, shuffle=True)
    xv, yv = torch.tensor(Xva), torch.tensor(yva)

    bester, zustand, t0 = float("inf"), None, time.time()
    for ep in range(epochen):
        netz.train()
        for xb, yb in lader:
            opt.zero_grad()
            verlust(netz(xb), yb).backward()
            nn.utils.clip_grad_norm_(netz.parameters(), 1.0)   # gegen exploding gradients
            opt.step()
        netz.eval()
        with torch.no_grad():
            vl = verlust(netz(xv), yv).item()
        if vl < bester:
            bester = vl
            zustand = {k: v.clone() for k, v in netz.state_dict().items()}
        print(f"  Epoche {ep+1:2d}/{epochen}   Val-MSE {vl:.4f}")

    netz.load_state_dict(zustand)
    netz.eval()
    with torch.no_grad():
        pred = netz(torch.tensor(Xte)).numpy()
    pred = scaler_y.inverse_transform(pred.reshape(-1, 1)).ravel()
    return netz, pred, time.time() - t0

In [ ]:
print("GRU")
netz_gru, pred_gru, zeit_gru = trainiere("gru")
y_test = d["y"].values[pos_te]
bewerte("GRU (48h Fenster)", y_test, pred_gru)
print(f"Trainingszeit: {zeit_gru:.0f} Sekunden")

In [ ]:
print("LSTM")
netz_lstm, pred_lstm, zeit_lstm = trainiere("lstm")
bewerte("LSTM (48h Fenster)", y_test, pred_lstm)
print(f"Trainingszeit: {zeit_lstm:.0f} Sekunden")

## 7. Der Vergleich

Jetzt alles vergleichen, inkl. der Trainingszeit.

In [ ]:
zeiten = {"Gradient Boosting": f"{zeit_gbm:.1f}s",
           "GRU (48h Fenster)":  f"{zeit_gru:.0f}s",
           "LSTM (48h Fenster)": f"{zeit_lstm:.0f}s"}

beste_baseline = min(ergebnisse["Naiv  y[t-1]"]["RMSE"],
                     ergebnisse["Saisonal-naiv  y[t-24]"]["RMSE"])

tabelle = pd.DataFrame(ergebnisse).T.round(1)
tabelle["RMSE besser als Baseline"] = [f"{100*(1 - r/beste_baseline):+.0f}%" for r in tabelle["RMSE"]]
tabelle["Trainingszeit"] = [zeiten.get(k, "-") for k in tabelle.index]
display(tabelle)

In [ ]:
fenster_plot = slice(600, 840)   # zehn Tage aus der Testperiode
plt.figure(figsize=(12, 4))
plt.plot(idx[pos_te][fenster_plot], y_test[fenster_plot], "k", lw=1.6, label="Beobachtet")
plt.plot(idx[pos_te][fenster_plot], gbm.predict(test[FEATURES])[-len(pos_te):][fenster_plot],
         lw=1.1, label="Gradient Boosting")
plt.plot(idx[pos_te][fenster_plot], pred_gru[fenster_plot], lw=1.1, label="GRU")
plt.legend(); plt.ylabel("Ausleihen / Stunde")
plt.title("Zehn Tage aus der Testperiode")
plt.tight_layout(); plt.show()

### Wie liest man diese Tabelle?

Drei Beobachtungen:

**1. Alle gelernten Modelle schlagen die Baseline deutlich.** Das ist die Mindesthürde,
und sie ist hier klar genommen. Wäre sie es nicht, müsste man das Modell wegwerfen,
egal wie modern es ist.

**2. Der Abstand zwischen Gradient Boosting und GRU ist klein.** Ein paar Prozent RMSE -
bei einem Vielfachen an Trainingszeit, mehr Code und mehr Dingen, die schiefgehen
können. Bei dieser Datenmenge und diesem Horizont ist das ein faires Rennen und kein
Triumph des Deep Learning.

**3. Das LSTM schlägt das GRU nicht.** Mehr Parameter sind nicht automatisch besser.
Deshalb die Praxisempfehlung aus der Vorlesung: mit GRU anfangen.

> **Die Frage für die Diskussion:** Welches Modell würdest du produktiv einsetzen?
> Die Antwort ist nicht automatisch das beste RMSE. Wartbarkeit, Trainingszeit,
> Nachvollziehbarkeit gegenüber Fachleuten und Robustheit gehören in die Abwägung.

### Wo RNNs wirklich gewinnen

Nicht bei dieser Aufgabe. Ihre Stärke spielen sie aus, wenn

* die Sequenz **lang und unregelmässig** ist und Lag-Features nicht reichen,
* mehrere Reihen gemeinsam modelliert werden (ein Modell für 1000 Filialen),
* der Input gar keine Tabelle ist - Text, Audio, Sensorströme,
* oder der Output selbst eine Sequenz ist.

Genau dort wird nächste Woche Attention ansetzen.

---

## 8. Optional: ein Netz, das Buchstaben erfindet

Dieser Teil ist freiwillig und dauert rund zwei Minuten. Er zeigt den **many-to-many**-Fall:
Input ist eine Zeichenkette, Output ist das jeweils nächste Zeichen.

Als Trainingstext erzeugen wir uns Sätze aus einem kleinen Baukasten. Das Netz sieht nur
einzelne Buchstaben - es weiss nichts von Wörtern, Leerzeichen oder Grammatik.

In [ ]:
import random
random.seed(SEED)

SUBJEKTE = ["das neuronale Netz", "der Gradient", "die Verlustfunktion", "das Modell",
            "die Baseline", "der verborgene Zustand", "die Lernrate", "das Fenster"]
VERBEN   = ["wächst", "schrumpft", "verschwindet", "stabilisiert sich", "überrascht uns",
            "verbessert sich", "schwankt stark", "konvergiert"]
ORTE     = ["im Training", "über die Zeitschritte", "nach wenigen Epochen", "auf den Testdaten",
            "bei langen Sequenzen", "ohne Skalierung", "mit Gradient Clipping"]

text = " ".join(
    f"{random.choice(SUBJEKTE)} {random.choice(VERBEN)} {random.choice(ORTE)}."
    for _ in range(2500))

zeichen = sorted(set(text))
zu_id = {c: i for i, c in enumerate(zeichen)}
zu_zeichen = {i: c for c, i in zu_id.items()}
print(f"{len(text)} Zeichen, Vokabular: {len(zeichen)}")
print(text[:180])

In [ ]:
L = 40   # Kontextlänge in Zeichen
daten = np.array([zu_id[c] for c in text], dtype=np.int64)
Xc = np.stack([daten[i:i+L]     for i in range(len(daten)-L-1)])
Yc = np.stack([daten[i+1:i+L+1] for i in range(len(daten)-L-1)])
Xc, Yc = Xc[:40000], Yc[:40000]

class ZeichenNetz(nn.Module):
    def __init__(self, V, emb=32, hidden=128):
        super().__init__()
        self.emb  = nn.Embedding(V, emb)
        self.rnn  = nn.GRU(emb, hidden, batch_first=True)
        self.kopf = nn.Linear(hidden, V)
    def forward(self, x, h=None):
        out, h = self.rnn(self.emb(x), h)
        return self.kopf(out), h     # Vorhersage für JEDEN Zeitschritt

torch.manual_seed(SEED)
zn  = ZeichenNetz(len(zeichen))
opt = torch.optim.Adam(zn.parameters(), lr=3e-3)
lader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(torch.tensor(Xc), torch.tensor(Yc)),
    batch_size=256, shuffle=True)

for ep in range(3):
    gesamt = 0.0
    for xb, yb in lader:
        opt.zero_grad()
        logits, _ = zn(xb)
        verlust = nn.functional.cross_entropy(logits.reshape(-1, len(zeichen)), yb.reshape(-1))
        verlust.backward(); opt.step()
        gesamt += verlust.item()
    print(f"Epoche {ep+1}  Loss {gesamt/len(lader):.3f}")

In [ ]:
@torch.no_grad()
def erfinde(start="das Modell", n=220, temperatur=0.6):
    zn.eval()
    h, ausgabe = None, start
    x = torch.tensor([[zu_id[c] for c in start]])
    for _ in range(n):
        logits, h = zn(x, h)
        p = torch.softmax(logits[0, -1] / temperatur, dim=-1)
        nächstes = int(torch.multinomial(p, 1))
        ausgabe += zu_zeichen[nächstes]
        x = torch.tensor([[nächstes]])
    return ausgabe

for t in [0.3, 0.7, 1.2]:
    print(f"--- Temperatur {t} ---")
    print(erfinde(temperatur=t), "\n")

### Was man hier sieht

Das Netz hat nie ein Wort gesehen, nur Buchstaben - und erzeugt trotzdem Wortgrenzen,
Umlaute an plausiblen Stellen und Satzpunkte. Es hat die Struktur aus der Statistik der
Zeichenfolgen gelernt.

Die **Temperatur** steuert, wie mutig gewürfelt wird. Niedrig heisst korrekt und
langweilig, hoch heisst kreativ und zunehmend unsinnig. Dasselbe Stellrad gibt es bei
heutigen Sprachmodellen - in Woche 11 kommt es wieder vor.

---

## 9. Zum Weiterarbeiten

1. **Fensterlänge.** Setze `SEQ` auf 12, dann auf 168, und trainiere neu. Wird es besser?
   Was passiert mit der Anzahl Trainingsbeispiele und der Trainingszeit?

2. **Der Horizont.** Baue das Ziel auf `y[t+24]` statt `y[t]` um. Wie verändert sich das
   Verhältnis zwischen naiver und saisonal-naiver Baseline - und warum?

3. **Das Leck bauen.** Nimm `temp` statt `temp_lag1` ins GBM auf und schau, was mit dem
   RMSE passiert. Erkläre in zwei Sätzen, warum dieses Modell im Betrieb nicht laufen
   könnte.

4. **Skalierungsfehler.** Fitte den `StandardScaler` absichtlich auf dem ganzen Datensatz
   statt nur auf Train. Wie gross ist der Effekt? Warum merkt man diesen Fehler so selten?

5. **Bidirektional.** Setze `bidirectional=True` im GRU (die Kopfschicht braucht dann
   `2*hidden` Eingänge). Das Resultat wird besser aussehen. Warum darfst du es trotzdem
   nicht als Prognoseergebnis berichten?

Die Lösungen zu 2 und 5 sind prüfungsnah.